# 03 — Content-Based Recommender

Builds a TF-IDF model over each movie's combined **genres + user tags** corpus and
uses cosine similarity (`linear_kernel`) to retrieve similar movies on demand.

**Optimisations applied**
- Removed duplicated EDA preprocessing (null checks, `dropna`) — done in notebook 01.
- Tags stripped of non-ASCII garbage bytes before vectorisation.
- `usecols` / dtype hints on CSV reads for memory efficiency.
- `pool_size = n * 10` gives a larger candidate buffer so the `min_ratings` filter
  still reliably yields ≥ n results.
- Single merged `content_movies` frame — no duplicate DataFrames.
- `sort=False` on `groupby` skips unnecessary sorting.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
movies = pd.read_csv(
    "../data/movies.csv",
    dtype={"movieId": "int32"},
    encoding="utf-8",
)

ratings = pd.read_csv(
    "../data/ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    usecols=["userId", "movieId", "rating"],
    encoding="utf-8",
)

tags = pd.read_csv(
    "../data/tags.csv",
    dtype={"movieId": "int32"},
    usecols=["movieId", "tag"],
    encoding="utf-8",
    errors="replace",
)

print(f"movies: {movies.shape}  ratings: {ratings.shape}  tags: {tags.shape}")

In [ ]:
# ── Tag preprocessing ────────────────────────────────────────────────────────
# 1. Drop the 17 null-tag rows.
# 2. Strip non-ASCII bytes that pollute the TF-IDF vocabulary.
tags_clean = tags.dropna(subset=["tag"]).copy()
tags_clean["tag"] = (
    tags_clean["tag"]
    .astype(str)
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
    .str.strip()
)

movie_tags = (
    tags_clean
    .groupby("movieId", sort=False)["tag"]
    .apply(" ".join)
    .reset_index()
)

movie_tags.head()

In [ ]:
# ── Rating stats ─────────────────────────────────────────────────────────────
rating_stats = (
    ratings
    .groupby("movieId", sort=False)
    .agg(avg_rating=("rating", "mean"), num_ratings=("rating", "count"))
    .reset_index()
)

In [ ]:
# ── Build single enriched content_movies frame ───────────────────────────────
content_movies = (
    movies
    .merge(movie_tags, on="movieId", how="left")
    .merge(rating_stats, on="movieId", how="left")
)

content_movies["tag"] = content_movies["tag"].fillna("")
content_movies["avg_rating"] = content_movies["avg_rating"].fillna(0.0)
content_movies["num_ratings"] = content_movies["num_ratings"].fillna(0).astype("int32")

# Combined content feature: genres + tag text.
content_movies["content"] = content_movies["genres"] + " " + content_movies["tag"]

content_movies.head()

In [ ]:
# ── TF-IDF matrix ────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(content_movies["content"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Map title → row position; keep first occurrence when duplicates exist.
indices = pd.Series(content_movies.index, index=content_movies["title"])
indices = indices[~indices.index.duplicated(keep="first")]

In [ ]:
def search_movie(query: str) -> pd.DataFrame:
    """Return up to 20 movies whose title contains `query` (case-insensitive)."""
    mask = content_movies["title"].str.contains(query, case=False, na=False, regex=False)
    return content_movies.loc[mask, ["title"]].head(20)

In [ ]:
def recommend_content(movie_title: str, n: int = 10, min_ratings: int = 50) -> pd.DataFrame:
    """Return n movies similar to `movie_title` by TF-IDF cosine similarity.

    A pool of n*10 candidates is fetched before applying the min_ratings filter
    so the result reliably contains n rows even for niche movies.
    """
    if movie_title not in indices:
        return pd.DataFrame({"Error": [f"'{movie_title}' not found in the catalogue."]})

    idx = indices[movie_title]

    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    pool_size = n * 10 + 1
    top_indices = sim_scores.argsort()[-pool_size:][::-1]
    top_indices = top_indices[top_indices != idx]  # exclude query film itself

    candidates = content_movies.iloc[top_indices][["title", "genres", "avg_rating", "num_ratings"]].copy()
    candidates["similarity"] = sim_scores[top_indices]

    candidates = candidates[candidates["num_ratings"] >= min_ratings]

    return (
        candidates
        .sort_values(["similarity", "avg_rating"], ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

In [ ]:
recommend_content("Toy Story (1995)")  # Test-1

In [ ]:
recommend_content("Matrix, The (1999)")  # Test-2

In [ ]:
recommend_content("Fight Club (1999)")  # Test-3

In [ ]:
recommend_content("Monsters, Inc. (2001)")  # Test-4

In [ ]:
os.makedirs("../outputs", exist_ok=True)
content_movies.to_csv("../outputs/content_based_movies.csv", index=False)
print("Saved → ../outputs/content_based_movies.csv")